# Start Here — Build a VSS AI Advisor

In this guided lab, you will deploy NVIDIA Video Search and Summarization (VSS) on an AWS Brev instance with **two RTX PRO 6000 GPUs**. You will inspect the hardware, reconstruct the model-to-GPU plan, render the container graph, deploy the stack, and prove that the models are running where you expect.

> **Outcome:** a working VSS AI Advisor that can ingest a short MP4, answer visual questions, cite useful timestamps, and generate a report.

Allow **15–25 minutes** for the first model download and initialization. The waiting time is part of the lab: you will use it to understand the services being started.

## Learning goals and journey

By the end of the workshop, you should be able to:

1. Explain why this Base topology gives the language and vision models separate GPUs.
2. Identify the NIM container, host GPU, port, and role for each model.
3. Read a rendered Docker Compose plan before starting it.
4. Distinguish image download, model initialization, container health, and application readiness.
5. Verify the deployment using Docker, NVIDIA telemetry, and HTTP health endpoints.
6. Use the advisor to turn video evidence into answers and reports.

| Phase | What you do | Evidence you collect |
| --- | --- | --- |
| Discover | Inspect GPUs and storage | Hardware inventory |
| Design | Map models to GPUs | Resolved topology |
| Plan | Render Compose services | Deployment graph |
| Deploy | Start NIMs, VSS, VIOS, and UI | Health-gated startup |
| Verify | Inspect containers, GPUs, and endpoints | Running-stack proof |
| Apply | Upload, ask, and report | Video AI result |

## Architecture: what you are building

```text
Browser
   │ HTTPS through Brev secure link (port 7777)
   ▼
HAProxy ingress
   ├── Workshop UI (Nginx, port 3100)
   ├── VSS Agent (port 8000)
   └── VIOS / VST video services (port 30888)
            │
            └── uploaded video, metadata, clips, and processing

VSS Agent
   ├── GPU 0 → Nemotron Nano 9B v2 NIM (language/orchestration, port 30081)
   └── GPU 1 → Cosmos3 Nano Reasoner NIM (visual reasoning, port 30082)
```

The **VSS Agent** coordinates the workflow. **Nemotron** interprets requests, selects tools, and shapes responses. **Cosmos3 Reasoner** performs visual reasoning over video. **VIOS/VST** manages video ingestion and processing. The workshop UI calls these services through one ingress, so attendees expose only port 7777.

The two NIMs are deliberately isolated on different host GPUs. That makes memory use and workshop performance more predictable than competing for one device.

## 0. Prepare the notebook workspace

Run this setup cell once. It finds the repository without assuming where Jupyter started, defines small display helpers, and points subsequent cells at the validated workshop runtime.

In [ ]:
from getpass import getpass
from pathlib import Path
from IPython.display import Markdown, display
import csv
import json
import os
import subprocess
import urllib.error
import urllib.request


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "workshop" / "scripts" / "deploy_vss_base.sh").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the vss-workshop repository in Jupyter.")


def read_env_file(path: Path) -> dict[str, str]:
    values = {}
    for line in path.read_text().splitlines():
        if line and not line.lstrip().startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            values[key] = value
    return values


def markdown_table(headers, rows):
    def clean(value):
        return str(value).replace("|", "&#124;").replace(chr(10), " ")

    header = "| " + " | ".join(clean(value) for value in headers) + " |"
    separator = "| " + " | ".join("---" for _ in headers) + " |"
    body = ["| " + " | ".join(clean(value) for value in row) + " |" for row in rows]
    display(Markdown(chr(10).join([header, separator, *body])))


REPO_ROOT = find_repo_root()
RUNTIME_DIR = REPO_ROOT / "workshop" / "runtime"
COMPOSE_FILE = RUNTIME_DIR / "compose.yml"
BASE_ENV_FILE = RUNTIME_DIR / "developer-profiles" / "dev-profile-base" / ".env"
DEPLOY_SCRIPT = REPO_ROOT / "workshop" / "scripts" / "deploy_vss_base.sh"

print(f"Workshop repository: {REPO_ROOT}")
print(f"Deployment entrypoint: {DEPLOY_SCRIPT}")

## 1. Discover the two GPUs

Before running the next cell, predict what you should see:

- How many GPUs should the VM expose?
- How much memory should each GPU have?
- Why might a language NIM and a vision NIM benefit from separate devices?

Now query the NVIDIA driver directly. The assertions are intentional lab guardrails: deployment should stop if this is not the expected workshop machine.

In [ ]:
gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ],
    check=True,
    capture_output=True,
    text=True,
)

gpu_inventory = [
    [field.strip() for field in row]
    for row in csv.reader(gpu_query.stdout.splitlines())
    if row
]

markdown_table(
    ["Host GPU", "GPU model", "Memory (MiB)", "Driver"],
    gpu_inventory,
)

assert len(gpu_inventory) == 2, f"Expected exactly two GPUs; found {len(gpu_inventory)}."
assert all("RTX PRO 6000" in row[1] for row in gpu_inventory), "This is not the validated workshop GPU type."
assert all(int(row[2]) >= 90_000 for row in gpu_inventory), "Each GPU needs at least 90,000 MiB."

print("Hardware checkpoint passed: the workshop has two suitable GPUs.")

### What the inventory tells you

The GPU index is the **host device ID** used by Docker Compose. It is not a model name and it is not a container-local CUDA index. In the next step, you will bind each NIM to one of these host IDs.

Memory being mostly empty now is expected. After deployment, rerun a focused telemetry cell and compare the result.

## 2. Reconstruct the model-to-GPU plan

This is the workshop's engineering decision point. Confirm the intended assignments below before running the cell. The cell compares your reconstruction with the version-controlled Base profile; it does not silently rewrite the deployment.

The validated choice is **one model per GPU**. Swapping the IDs would still isolate the models, but every attendee uses the same mapping so troubleshooting and demonstrations remain consistent.

In [ ]:
# Learning task: confirm these host GPU IDs before running the cell.
workshop_gpu_plan = {
    "LLM_DEVICE_ID": "0",  # Nemotron: language, orchestration, and response generation
    "VLM_DEVICE_ID": "1",  # Cosmos3 Reasoner: visual understanding
}

assert len(set(workshop_gpu_plan.values())) == 2, "Assign the LLM and VLM to different GPUs."

base_profile = read_env_file(BASE_ENV_FILE)
for setting, planned_gpu in workshop_gpu_plan.items():
    configured_gpu = base_profile.get(setting)
    assert planned_gpu == configured_gpu, (
        f"Your {setting} plan uses GPU {planned_gpu}, but the validated profile uses GPU {configured_gpu}."
    )

model_plan = [
    [
        "Language NIM",
        base_profile["LLM_NAME"],
        workshop_gpu_plan["LLM_DEVICE_ID"],
        base_profile["LLM_PORT"],
        "Interpret requests and orchestrate the answer",
    ],
    [
        "Vision NIM",
        base_profile["VLM_NAME"],
        workshop_gpu_plan["VLM_DEVICE_ID"],
        base_profile["VLM_PORT"],
        "Reason over video content",
    ],
]

markdown_table(["Role", "Model", "Host GPU", "Host port", "Responsibility"], model_plan)
print("Topology checkpoint passed: the two model services reserve different GPUs.")

## 3. Render the deployment plan before starting it

Docker Compose combines the root runtime file, service fragments, Base profile, hardware profile, and GPU reservations into one resolved graph. Rendering it first catches missing variables and shows exactly which images and dependencies will be used.

The placeholder NGC value below is used only to render configuration. It cannot authenticate and is never saved.

In [ ]:
compose_environment = os.environ.copy()
compose_environment.update(
    {
        "VSS_APPS_DIR": str(RUNTIME_DIR),
        "VSS_DATA_DIR": "/tmp/vss-workshop-data",
        "HOST_IP": "127.0.0.1",
        "EXTERNAL_IP": "127.0.0.1",
        "VSS_PUBLIC_HOST": "127.0.0.1",
        "NGC_CLI_API_KEY": "not-used-for-rendering",
    }
)

compose_render = subprocess.run(
    [
        "docker",
        "compose",
        "--env-file",
        str(BASE_ENV_FILE),
        "-f",
        str(COMPOSE_FILE),
        "config",
        "--format",
        "json",
    ],
    check=True,
    capture_output=True,
    text=True,
    env=compose_environment,
)
resolved_compose = json.loads(compose_render.stdout)
resolved_services = resolved_compose["services"]


def reserved_gpu_ids(service):
    devices = (
        service.get("deploy", {})
        .get("resources", {})
        .get("reservations", {})
        .get("devices", [])
    )
    ids = [device_id for device in devices for device_id in device.get("device_ids", [])]
    return ", ".join(ids) if ids else "—"


focus_services = [
    "nvidia-nemotron-nano-9b-v2",
    "cosmos3-reasoner",
    "vss-agent",
    "sensor-ms",
    "streamprocessing-ms",
    "vst-ingress",
    "redis",
    "centralizedb",
    "vss-ui",
    "vss-haproxy-ingress",
]

service_rows = []
for service_name in focus_services:
    service = resolved_services.get(service_name)
    if not service:
        continue
    dependencies = ", ".join(service.get("depends_on", {}).keys()) or "—"
    service_rows.append(
        [service_name, service.get("image", "—"), reserved_gpu_ids(service), dependencies]
    )

markdown_table(["Service", "Image", "Host GPU", "Waits for"], service_rows)
print(f"Compose rendered {len(resolved_services)} active services without starting containers.")

### Read the graph

Notice the dependency chain:

1. The two NIMs initialize independently on GPU 0 and GPU 1.
2. The VSS Agent waits for both model health checks.
3. Video services provide ingestion and processing.
4. The repository-owned UI starts after the agent is healthy.
5. HAProxy presents the UI and APIs through port 7777.

**Prediction checkpoint:** which two services are likely to take longest on the first run, and why?

## 4. Run the full preflight

You inspected the educational parts manually; now run the deployment's authoritative guardrails. They validate:

- exactly two RTX PRO 6000 GPUs and the NVIDIA driver;
- Docker Engine and Compose compatibility;
- free Docker and ephemeral storage;
- the complete Base Compose graph.

If Docker is newer than the validated range, preflight may report a managed repair. The deploy step can pin it on a clean workshop VM. Treat every other failure as a real blocker.

In [ ]:
subprocess.run([str(DEPLOY_SCRIPT), "check"], check=True)

## 5. Add your NGC API key privately

The key authorizes Docker to pull NVIDIA containers and allows NIM to obtain protected model artifacts. It does **not** assign GPUs or expose the UI.

Create or find your key at [NGC API Keys](https://org.ngc.nvidia.com/setup/api-keys). Paste it only into the hidden prompt below. Never place it in notebook source, chat, screenshots, or Git.

In [ ]:
if not os.environ.get("NGC_API_KEY", "").strip():
    os.environ["NGC_API_KEY"] = getpass("Paste your NGC API key (input is hidden): ").strip()

if not os.environ["NGC_API_KEY"]:
    raise ValueError("An NGC API key is required for this workshop.")

print("Key is available to this kernel. It remains hidden and will be passed privately to deployment.")

## 6. Deploy the Base stack

This is the only long-running cell. While it runs, connect each message to a deployment phase:

| Phase | What happens |
| --- | --- |
| Host preparation | Pin Docker if required; choose large storage; prepare writable Redis/VIOS paths |
| Authentication | Send the NGC key to NVCR through password stdin |
| Artifact pull | Download container layers and model-serving images; detailed progress goes to a local log |
| Scheduling | Reserve GPU 0 for Nemotron and GPU 1 for Cosmos3 Reasoner |
| Initialization | Load model artifacts and start NIM endpoints |
| Dependency startup | Start VSS Agent, video services, UI, and ingress after their prerequisites are healthy |

Detailed image-layer output is intentionally kept out of Jupyter. If you want to observe it, open a terminal and run:

```bash
tail -f ~/.local/state/vss-workshop/deploy.log
```

Keep the next cell running. First deployment typically takes **15–25 minutes**.

In [ ]:
deployment_environment = os.environ.copy()
subprocess.run(
    [str(DEPLOY_SCRIPT), "deploy"],
    check=True,
    env=deployment_environment,
)

## 7. Verify what was deployed

A successful `deploy` means the scripted health gates passed. Do not stop there: collect independent evidence from Compose, Docker's GPU reservations, NVIDIA telemetry, and the HTTP endpoints.

### 7.1 Check service state

The status action is safe at any time. Look for healthy model containers, a healthy agent, video services, the workshop UI, and HAProxy.

In [ ]:
subprocess.run([str(DEPLOY_SCRIPT), "status"], check=True)

### 7.2 Prove which host GPU each NIM received

The Compose plan was intent. `docker inspect` is runtime evidence. Docker's `DeviceRequests` should show GPU 0 for Nemotron and GPU 1 for Cosmos3 Reasoner.

In [ ]:
model_containers = [
    ("nvidia-nemotron-nano-9b-v2", "Language NIM"),
    ("nvidia-cosmos3-reasoner", "Vision NIM"),
]

assignment_rows = []
for container_name, role in model_containers:
    inspected = subprocess.run(
        ["docker", "inspect", container_name],
        check=True,
        capture_output=True,
        text=True,
    )
    container = json.loads(inspected.stdout)[0]
    requests = container["HostConfig"].get("DeviceRequests") or []
    device_ids = [device_id for request in requests for device_id in (request.get("DeviceIDs") or [])]
    health = container["State"].get("Health", {}).get("Status", container["State"]["Status"])
    assignment_rows.append([container_name, role, ", ".join(device_ids) or "—", health])

markdown_table(["Container", "Role", "Reserved host GPU", "Runtime health"], assignment_rows)

actual_assignments = {row[1]: row[2] for row in assignment_rows}
assert actual_assignments["Language NIM"] == "0", "Nemotron is not reserved on host GPU 0."
assert actual_assignments["Vision NIM"] == "1", "Cosmos3 Reasoner is not reserved on host GPU 1."
print("Runtime assignment checkpoint passed.")

### 7.3 Observe GPU memory after model initialization

Compare this output with the initial inventory. Device reservation proves placement; memory use shows that model-serving processes have loaded work onto the GPUs. Utilization may be low while the workshop is idle.

In [ ]:
runtime_gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
        "--format=csv,noheader,nounits",
    ],
    check=True,
    capture_output=True,
    text=True,
)

runtime_gpu_rows = [
    [field.strip() for field in row]
    for row in csv.reader(runtime_gpu_query.stdout.splitlines())
    if row
]
markdown_table(
    ["Host GPU", "GPU model", "Used memory (MiB)", "Total memory (MiB)", "GPU utilization (%)"],
    runtime_gpu_rows,
)

### 7.4 Probe the model and application endpoints

Container state alone does not prove an application can answer requests. These local probes test both NIM readiness endpoints, the VSS Agent, the custom workshop UI, and the shared ingress.

In [ ]:
health_endpoints = [
    ("Nemotron NIM", "http://127.0.0.1:30081/v1/health/ready"),
    ("Cosmos3 Reasoner NIM", "http://127.0.0.1:30082/v1/health/ready"),
    ("VSS Agent", "http://127.0.0.1:8000/health"),
    ("Workshop UI", "http://127.0.0.1:3100/health"),
    ("HAProxy ingress", "http://127.0.0.1:7777/"),
]

health_rows = []
for component, url in health_endpoints:
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            health_rows.append([component, url, response.status, "ready"])
    except (urllib.error.URLError, TimeoutError) as error:
        health_rows.append([component, url, "—", f"not ready: {error}"])

markdown_table(["Component", "Local endpoint", "HTTP", "Result"], health_rows)

if all(row[3] == "ready" for row in health_rows):
    print("Application readiness checkpoint passed: every workshop endpoint responded.")
else:
    print("One or more endpoints need attention. Run the status cell and inspect the named container.")

## 8. Open the workshop UI

In Brev, open or create a **secure link for port 7777 only**. Do not expose the internal NIM, agent, database, Redis, or VIOS ports. The URL printed by `status` should follow this pattern:

```text
https://ui-<brev-instance-id>.brevlab.com
```

The persistent **AI Video Advisor** is on the left. The video workspace and **Upload video** action are on the right.

## 9. Apply the system: upload, ask, and report

Use a short H.264 MP4 and complete this evidence-driven sequence:

1. Upload the file and wait until it appears in the video workspace.
2. Select its card. Confirm that the left panel shows it as the current video.
3. Ask for a concise timeline. Check whether the answer includes useful timestamps.
4. Ask a focused visual question: `At what timestamp does the main action begin, and what visual evidence supports that answer?`
5. Generate a report that distinguishes observed facts from inference.

The important learning step is not merely receiving an answer—it is testing whether the answer is grounded in the selected video.

### Prompt experiment

Try the same video with three levels of specificity and compare the results:

| Prompt style | Example | What to evaluate |
| --- | --- | --- |
| Broad | `Summarize this video.` | Coverage and omissions |
| Temporal | `Create a timeline with timestamps.` | Time grounding |
| Evidence-led | `Identify the key action, give its timestamp, and describe the visual evidence.` | Precision and support |

Discuss with a partner: which prompt produced the most defensible answer, and why?

## 10. Knowledge check

Answer these before revealing the explanations:

1. Where is the GPU assignment declared, and how did you verify the runtime assignment?
2. Why does the VSS Agent wait for both NIMs?
3. What is the NGC key used for—and what is it not used for?
4. Why is port 7777 the only Brev secure link needed?
5. What is the difference between a running container and a ready application?

In [ ]:
show_explanations = False  # Change to True after discussing the questions.

explanations = [
    "Compose reserves LLM_DEVICE_ID=0 and VLM_DEVICE_ID=1; docker inspect verifies DeviceRequests at runtime.",
    "The agent depends on both language orchestration and visual reasoning, so it starts only after both NIM health checks pass.",
    "The key authenticates NVIDIA container/model access. It does not choose GPUs, open ports, or authenticate workshop attendees.",
    "HAProxy routes the UI and required APIs behind one public endpoint; internal service ports remain local to the VM.",
    "Running means the process exists. Ready means its health endpoint can serve the expected application workload.",
]

if show_explanations:
    for number, explanation in enumerate(explanations, start=1):
        print(f"{number}. {explanation}")
else:
    print("Discuss your answers first, then set show_explanations = True.")

## 11. Stop safely when finished

The stop action preserves uploaded videos, Docker volumes, model caches, and VM-local deployment state. This makes a later restart faster. It does not stop Brev billing by itself.

After stopping the services, stop the Brev VM from the Brev console to stop GPU billing. Terminate the VM only when you no longer need its data.

In [ ]:
stop_vss_now = False  # Change to True only when you are ready to stop workshop services.

if stop_vss_now:
    subprocess.run([str(DEPLOY_SCRIPT), "stop"], check=True)
else:
    print("VSS is still running. Change stop_vss_now to True when you are finished.")

## What you built

You moved through the same reasoning an operator uses in a real GPU deployment:

- **inventory** the accelerator hardware;
- **design** a model-to-GPU topology;
- **render** infrastructure before changing state;
- **deploy** with dependency and health gates;
- **verify** placement, memory use, and application readiness;
- **apply** the system to an evidence-grounded video task;
- **stop** compute without deleting workshop data.

As a next exercise, inspect `workshop/runtime/services/nim/` and find the exact Compose `device_ids` reservation for each NIM. Then trace the VSS Agent's model endpoints in `workshop/runtime/services/agent/compose.yml`.